<!-- Copyright (c) 2026 Robotics and AI Institute LLC dba RAI Institute. All rights reserved. -->

# SAM3 Dataset Creation

This notebook creates a segmentation dataset from images using SAM3.
It processes images from a source directory and generates masks using text prompts,
then saves them in the same format as the training datasets.

In [ ]:
import sys
from pathlib import Path

# Add the scripts directory to path
notebook_dir = Path().resolve()
scripts_dir = notebook_dir.parent.parent
sys.path.insert(0, str(scripts_dir))

import torch
from PIL import Image
import numpy as np
from tqdm import tqdm
import random

from sem_map.perception.models.sam3 import (
    load_sam3_model,
    process_multiple_prompts,
    get_semantic_segmentation_tensor,
)
from sem_map.perception.common import get_workspace_root

# Get workspace root for resolving relative paths
WORKSPACE_ROOT = get_workspace_root()
print(f"Workspace root: {WORKSPACE_ROOT}")

In [ ]:
# Configuration
# Paths are relative to workspace root
SOURCE_IMAGE_DIR = WORKSPACE_ROOT / "data/custom_datasets/indoor_racing_white/001/images"
OUTPUT_BASE_DIR = WORKSPACE_ROOT / "data/custom_datasets"
DATASET_NAME = "indoor_racing_white"  # e.g., "indoor_racing", "outdoor_trail", etc.
DATASET_INDEX = "002"  # e.g., "001", "002", etc.

# Optional: path to a text file listing selected images to process (one path per line)
# Set to None to process all images in SOURCE_IMAGE_DIR
# e.g., "data/custom_datasets/indoor_racing_white/001/images/selected_kmeans_k10.txt"
SELECTED_IMAGES_FILE = "data/custom_datasets/my_scene/001/images/selected_kmeans_k20.txt"  # or WORKSPACE_ROOT / "path/to/selected_kmeans_k10.txt"

# Text prompts and their corresponding class IDs
# Format: {prompt_text: class_id}
PROMPTS_TO_CLASS_IDS = {
    "indoor floor": 2,  # traversable floor
    "white floor": 1,  # non-traversable floor
}

# Background class ID (pixels not matching any prompt)
BACKGROUND_CLASS_ID = 0

# Priority order for overlapping masks (higher priority overwrites lower)
# Class IDs listed first have higher priority
CLASS_ID_PRIORITY = [1, 2]  # e.g., class 1 has higher priority than class 2

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Load SAM3 model (HuggingFace backend)
print("Loading SAM3 model...")
predictor = load_sam3_model()
print("Model loaded successfully!")

In [ ]:
# Get list of image files
if SELECTED_IMAGES_FILE is not None:
    # Load selected images from text file (one path per line, absolute or relative to workspace)
    selected_file = Path(SELECTED_IMAGES_FILE)
    if not selected_file.exists():
        raise FileNotFoundError(f"Selected images file not found: {selected_file}")
    
    image_files = []
    for line in selected_file.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        p = Path(line)
        if not p.is_absolute():
            p = WORKSPACE_ROOT / p
        if not p.exists():
            print(f"Warning: selected image not found, skipping: {p}")
            continue
        image_files.append(p)
    image_files = sorted(image_files)
    print(f"Loaded {len(image_files)} images from {selected_file.name}")
else:
    # Use all images in source directory
    source_dir = Path(SOURCE_IMAGE_DIR).resolve()
    if not source_dir.exists():
        raise FileNotFoundError(f"Source directory not found: {source_dir}\nPlease check the path in the configuration cell.")

    image_extensions = {'.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG'}
    image_files = sorted([f for f in source_dir.iterdir() 
                          if f.suffix in image_extensions])
    print(f"Found {len(image_files)} images in {source_dir}")

if len(image_files) > 0:
    print(f"First image: {image_files[0].name}")
    print(f"Last image: {image_files[-1].name}")
else:
    print("No images found!")

In [ ]:
def combine_masks_to_segmentation(
    masks_dict: dict,
    prompts_to_class_ids: dict,
    class_id_priority: list,
    background_class_id: int,
    image_size: tuple,
) -> torch.Tensor:
    """Combine multiple SAM3 masks into a single segmentation mask.
    
    Args:
        masks_dict: Dictionary mapping prompt text to semantic segmentation tensor
        prompts_to_class_ids: Mapping from prompt text to class ID
        class_id_priority: List of class IDs in priority order (highest first)
        background_class_id: Class ID for background pixels
        image_size: (width, height) tuple
    
    Returns:
        Segmentation mask tensor of shape (H, W) with class IDs
    """
    h, w = image_size[1], image_size[0]
    
    # Initialize segmentation mask with background
    segmentation = torch.full((h, w), background_class_id, dtype=torch.uint8)
    
    # Create mapping from class_id to priority index (lower index = higher priority)
    priority_map = {class_id: idx for idx, class_id in enumerate(class_id_priority)}
    
    # Sort prompts by priority: lowest priority first, so highest priority is painted last and wins
    prompts_sorted = sorted(
        masks_dict.keys(),
        key=lambda p: priority_map.get(prompts_to_class_ids.get(p, float('inf')), float('inf')),
        reverse=True
    )
    
    # Process masks: lowest priority first, highest priority last (overwrites on overlap)
    for prompt in prompts_sorted:
        if prompt not in masks_dict:
            continue
            
        result = masks_dict[prompt]
        class_id = prompts_to_class_ids[prompt]
        
        # Get semantic segmentation mask
        mask_tensor = get_semantic_segmentation_tensor(
            result['semantic_seg'],
            h, w,
            threshold=0.5
        )
        
        # Apply mask: set pixels where mask > 0.5 to class_id
        mask_binary = (mask_tensor > 0.5).cpu()
        segmentation[mask_binary] = class_id
    
    return segmentation

In [ ]:
# Create output directories
output_dir = OUTPUT_BASE_DIR / DATASET_NAME / DATASET_INDEX
images_dir = output_dir / "images"
segmentations_dir = output_dir / "segmentations"

images_dir.mkdir(parents=True, exist_ok=True)
segmentations_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir}")
print(f"Images will be saved to: {images_dir}")
print(f"Masks will be saved to: {segmentations_dir}")

In [ ]:
# Process all images
text_prompts = list(PROMPTS_TO_CLASS_IDS.keys())

print(f"Processing {len(image_files)} images with prompts: {text_prompts}")
print(f"Class IDs: {PROMPTS_TO_CLASS_IDS}")

failed_images = []

for idx, image_path in enumerate(tqdm(image_files, desc="Processing images")):
    try:
        # Load image
        image = Image.open(image_path).convert("RGB")
        
        # Process with SAM3
        results = process_multiple_prompts(
            predictor, image, text_prompts
        )
        
        # Combine masks into segmentation
        segmentation = combine_masks_to_segmentation(
            results,
            PROMPTS_TO_CLASS_IDS,
            CLASS_ID_PRIORITY,
            BACKGROUND_CLASS_ID,
            image.size
        )
        
        # Save image
        output_image_name = f"{idx:06d}.png"
        image.save(images_dir / output_image_name)
        
        # Save segmentation mask
        segmentation_np = segmentation.cpu().numpy().astype(np.uint8)
        Image.fromarray(segmentation_np, mode='L').save(
            segmentations_dir / output_image_name
        )
        
    except Exception as e:
        print(f"\nError processing {image_path.name}: {e}")
        failed_images.append(image_path.name)
        continue

print(f"\nProcessed {len(image_files) - len(failed_images)}/{len(image_files)} images successfully")
if failed_images:
    print(f"Failed images: {failed_images}")

## Rerun Visualization

Stream the created dataset to Rerun for visualization and inspection.

In [ ]:
import rerun as rr
from typing import List

# Monkey-patch numpy.asarray to handle 'copy' argument for Rerun compatibility
# NumPy 1.26+ doesn't support 'copy' argument, but Rerun 0.23.1 tries to use it
_original_asarray = np.asarray
def _patched_asarray(a, dtype=None, order=None, *, copy=None, **kwargs):
    """Patched np.asarray that handles 'copy' argument for compatibility with Rerun"""
    if copy is True:
        # If copy=True, ensure we return a copy
        result = np.array(a, dtype=dtype, order=order, copy=True, **kwargs)
    elif copy is False:
        # If copy=False, try to avoid copying (but asarray always copies if needed)
        result = _original_asarray(a, dtype=dtype, order=order, **kwargs)
    else:
        # No copy argument, use original behavior
        result = _original_asarray(a, dtype=dtype, order=order, **kwargs)
    return result
np.asarray = _patched_asarray

def hex_to_rgb(hex_color: str) -> List[int]:
    hex_color = hex_color.lstrip("#")
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return [r, g, b]

# Create annotation info for each class
label_infos = []

# Add background class
label_infos.append(rr.AnnotationInfo(id=BACKGROUND_CLASS_ID, label="background", color=[0, 0, 0]))

# Add classes from prompts
# Generate colors for each class (using tab20 colormap)
import matplotlib.cm as cm
tab20 = cm.get_cmap('tab20')

for prompt, class_id in sorted(PROMPTS_TO_CLASS_IDS.items(), key=lambda x: x[1]):
    # Get color from tab20 colormap
    color_rgba = tab20(class_id / max(len(PROMPTS_TO_CLASS_IDS), 1))
    color_rgb = [int(c * 255) for c in color_rgba[:3]]
    label_infos.append(rr.AnnotationInfo(id=class_id, label=prompt, color=color_rgb))

annotation_ctx = rr.AnnotationContext(label_infos)
annotation_ctx

In [ ]:
# Initialize Rerun
recording_id = f"sam3_dataset_{DATASET_NAME}_{DATASET_INDEX}"
rr.init("sam3_dataset_creation", recording_id=recording_id)
rr.connect_grpc()
rr.log(f"{DATASET_NAME}/mask", annotation_ctx, static=True)

MAX_FRAMES = None  # Set to an int to limit how many frames we stream

# Use sorted order for Rerun (original sequence, not shuffled train/val order)
image_names_for_rerun = sorted([f.stem for f in images_dir.glob("*.png")])
for frame_idx, image_name in enumerate(image_names_for_rerun):
    if MAX_FRAMES is not None and frame_idx >= MAX_FRAMES:
        break

    # Load image
    image_path = images_dir / f"{image_name}.png"
    rgb = np.array(Image.open(image_path).convert("RGB"))

    # Load segmentation
    seg_path = segmentations_dir / f"{image_name}.png"
    mask = np.array(Image.open(seg_path))

    # Set time and log
    rr.set_time(timeline="frame", sequence=frame_idx)
    rr.log(f"{DATASET_NAME}/raw", rr.Image(rgb))
    rr.log(f"{DATASET_NAME}/mask/segmentation", rr.SegmentationImage(mask))

print("Done streaming frames to Rerun.")

## Train and Val Split

In [ ]:
# Create train/val splits
train_ratio = 0.8
seed = 42

# Get all image names (without extension)
all_image_names = sorted([f.stem for f in images_dir.glob("*.png")])

# Omit images listed in invalid_masks.txt if it exists
invalid_masks_path = output_dir / "invalid_masks.txt"
if invalid_masks_path.exists():
    invalid_stems = {line.strip() for line in invalid_masks_path.read_text().splitlines() if line.strip()}
    all_image_names = [n for n in all_image_names if n not in invalid_stems]
    print(f"Excluded {len(invalid_stems)} invalid masks")

if len(all_image_names) == 0:
    print("No images found to create splits!")
else:
    # Shuffle with fixed seed
    random.Random(seed).shuffle(all_image_names)
    
    # Split
    split_idx = int(len(all_image_names) * train_ratio)
    train_ids = sorted(all_image_names[:split_idx])
    val_ids = sorted(all_image_names[split_idx:])
    
    # Save splits
    train_list_path = output_dir / "train_images.txt"
    val_list_path = output_dir / "val_images.txt"
    
    train_list_path.write_text("\n".join(train_ids))
    val_list_path.write_text("\n".join(val_ids))
    
    print(f"Created {train_list_path} with {len(train_ids)} entries")
    print(f"Created {val_list_path} with {len(val_ids)} entries")